# Processing Clinical Notes with the Anthropic API
### Turning free text into structured data

**Python for Data Science - Working with Unstructured Text**

---

In the encoding and binning notebooks, every column arrived clean: `sex`, `age`, `class`. Real clinical text does not. A discharge note is a paragraph of prose - no columns, nothing to `groupby`, nothing to encode.

This notebook is the missing first step. We take short clinical notes and use a **large language model, through the Anthropic API**, to turn them into a structured table you could then encode and bin like any other dataset.

We will do four things to each note:

| Task | Turns a note into... |
|---|---|
| **Summarize (clinician)** | a one-line clinical summary |
| **Explain (patient)** | a plain-language version with no jargon |
| **Classify** | one department from a fixed list |
| **Check consistency** | a flag: does the note match its stated diagnosis? |

By the end, a pile of prose has become a tidy dataframe.

> **The data here is synthetic and de-identified on purpose.** Every note was written by hand for this exercise, describes a fictional patient, and contains no names, dates of birth, or record numbers. That is deliberate: before you ever send clinical text to an external service, you strip anything that could identify a real person. We will come back to this.

---

## 1. The notes

Eight short notes, in the style of an emergency triage or discharge summary. Each carries a **stated diagnosis** (we will check it later) and a `department_reference` - the department we intended when writing it. In real life you would not have that reference column; it is here only because we wrote the notes ourselves.

In [7]:
import pandas as pd

NOTES = [
    {"note_id": "n01",
     "text": ("64-year-old man with 2 hours of crushing substernal chest pain radiating "
              "to the left arm, with sweating and nausea. ECG shows ST elevation in the "
              "inferior leads and troponin is markedly raised. Given aspirin and heparin "
              "and taken for urgent cardiac catheterization."),
     "stated_diagnosis": "ST-elevation myocardial infarction",
     "department_reference": "Cardiology"},

    {"note_id": "n02",
     "text": ("71-year-old woman with known COPD and three days of worsening breathlessness, "
              "wheeze, and a productive cough of green sputum. Using accessory muscles, "
              "oxygen saturation 86% on room air. Treated with nebulized bronchodilators, "
              "steroids, and controlled oxygen."),
     "stated_diagnosis": "Acute exacerbation of COPD",
     "department_reference": "Respiratory"},

    {"note_id": "n03",
     "text": ("48-year-old man with weeks of burning epigastric pain, worse at night and "
              "eased by antacids, now passing black tarry stools. Hemoglobin is low. "
              "Endoscopy shows a bleeding gastric ulcer, treated with a proton pump "
              "inhibitor infusion and endoscopic clipping."),
     "stated_diagnosis": "Bleeding peptic ulcer",
     "department_reference": "Gastroenterology"},

    {"note_id": "n04",
     "text": ("77-year-old woman with sudden right-sided facial droop, arm weakness, and "
              "slurred speech starting 90 minutes ago. CT excludes hemorrhage. Assessed "
              "for thrombolysis and admitted to the stroke unit."),
     "stated_diagnosis": "Acute ischemic stroke",
     "department_reference": "Neurology"},

    {"note_id": "n05",
     "text": ("22-year-old woman with a day of vomiting, deep rapid breathing, and "
              "drowsiness. Known type 1 diabetes, glucose very high, ketones present, "
              "and a blood gas showing metabolic acidosis. Started on intravenous fluids "
              "and an insulin infusion."),
     "stated_diagnosis": "Diabetic ketoacidosis",
     "department_reference": "Endocrinology"},

    {"note_id": "n06",
     "text": ("35-year-old man with sudden severe left flank pain radiating to the groin, "
              "unable to keep still, with visible blood in the urine. CT shows a 6 mm stone "
              "in the left ureter. Given analgesia and fluids and referred to urology."),
     "stated_diagnosis": "Ureteric stone",
     "department_reference": "Urology"},

    # --- the next two carry a WRONG stated diagnosis on purpose ---
    {"note_id": "n07",
     "text": ("19-year-old man with a day of pain that began around the umbilicus and moved "
              "to the right lower abdomen, with fever, nausea, and marked tenderness and "
              "guarding in the right iliac fossa. White cell count raised. Surgical team "
              "consulted for likely appendicitis."),
     "stated_diagnosis": "Migraine headache",
     "department_reference": "Gastroenterology"},

    {"note_id": "n08",
     "text": ("68-year-old man with four days of fever, shaking chills, and a productive "
              "cough of rusty sputum, now short of breath. Crackles at the right lung base "
              "and consolidation on chest X-ray. Started on antibiotics for "
              "community-acquired pneumonia."),
     "stated_diagnosis": "Ankle sprain",
     "department_reference": "Respiratory"},
]

notes = pd.DataFrame(NOTES)
print(f"{len(notes)} notes loaded")
notes[["note_id", "stated_diagnosis", "department_reference"]]

8 notes loaded


,note_id,stated_diagnosis,department_reference
0,n01,ST-elevation myocardial infarction,Cardiology
1,n02,Acute exacerbation of COPD,Respiratory
2,n03,Bleeding peptic ulcer,Gastroenterology
3,n04,Acute ischemic stroke,Neurology
4,n05,Diabetic ketoacidosis,Endocrinology
5,n06,Ureteric stone,Urology
6,n07,Migraine headache,Gastroenterology
7,n08,Ankle sprain,Respiratory


In [8]:
# Read one in full, to see what we are working with.
print(notes.loc[0, "text"])

64-year-old man with 2 hours of crushing substernal chest pain radiating to the left arm, with sweating and nausea. ECG shows ST elevation in the inferior leads and troponin is markedly raised. Given aspirin and heparin and taken for urgent cardiac catheterization.


Notice what these notes are *not*: there is no `age` column, no `diagnosis` column, no way to filter or count anything. The information is all there, locked inside prose. Extracting it is the job.

---

## 2. Connecting to the API

We use Anthropic's **Messages API** through the official `anthropic` Python package.

### One-time setup

```bash
pip install anthropic
```

Then set your key as an **environment variable** - never paste it into the notebook, because notebooks get shared and committed to git:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

### On cost and privacy

- We use **Claude Haiku**, the smallest and cheapest model. Eight notes times three tasks is a couple of dozen short calls - a fraction of a cent in total.
- The notes contain **no identifiers**, by design. That is the rule for real work too: de-identify before any text leaves your machine. A patient's name adds nothing to a summarization task and everything to your risk.

In [9]:
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")




import anthropic

MODEL = "claude-haiku-4-5-20251001"   # model names change over time; see docs.anthropic.com

client = anthropic.Anthropic()        # reads ANTHROPIC_API_KEY from the environment
print("client ready, using", MODEL)

client ready, using claude-haiku-4-5-20251001


### The one function we will reuse

Every task below is the same shape: a **system prompt** (the standing instruction - who the model is and what format to reply in) and a **user message** (the note itself). We wrap that in a single helper.

Two arguments worth understanding:

- `temperature=0` asks for the most deterministic answer the model can give. For a data pipeline you want the same input to produce the same output, so you keep it at 0. Higher values add randomness, which is for creative work, not extraction.
- `max_tokens` caps the length of the reply. Our answers are short, so a small cap keeps things fast and cheap.

In [10]:
def call_claude(system, user, max_tokens=200, temperature=0):
    """Send one system+user message to Claude and return the text of the reply."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    return response.content[0].text

In [11]:
# Always test with a single cheap call before looping over your whole dataset -
# it confirms the key, the network, and the model name are all working.
print(call_claude(system="Reply with exactly one word: pong.",
                  user="ping"))

pong


In [25]:
pd.options.display.max_colwidth = 200

---

## 3. Task 1 - Summarize each note (for a clinician)

The gentlest task: free text in, free text out. The only skill is writing a **system prompt** that pins down the format you want - here, one short sentence and no invented details.

In [26]:
SUMMARY_SYSTEM = (
    "You are a clinical documentation assistant writing for a colleague. "
    "Summarize the clinical note in ONE concise sentence of at most 20 words, "
    "using standard clinical terminology. "
    "Do not add information that is not in the note, and do not include names or identifiers."
)

def summarize(note_text):
    return call_claude(SUMMARY_SYSTEM, note_text, max_tokens=80).strip()

notes["llm_summary"] = notes["text"].apply(summarize)

notes[["note_id", "llm_summary"]]

,note_id,llm_summary
0,n01,"Acute inferior STEMI treated with aspirin, heparin, and urgent cardiac catheterization."
1,n02,"COPD exacerbation with hypoxemia treated with bronchodilators, steroids, and supplemental oxygen."
2,n03,"Bleeding gastric ulcer with melena and anemia, treated with PPI infusion and endoscopic clipping."
3,n04,"Acute ischemic stroke with facial droop, arm weakness, and dysarthria; thrombolysis candidate admitted to stroke unit."
4,n05,Type 1 diabetic with diabetic ketoacidosis treated with IV fluids and insulin infusion.
5,n06,"Acute left ureteral calculus with hematuria treated with analgesia, hydration, and urology referral."
6,n07,Acute appendicitis with peritoneal signs; elevated WBC; surgical consultation obtained.
7,n08,"68-year-old with community-acquired pneumonia presenting with fever, productive cough, dyspnea, and right lower lobe consolidation."


That summary is written for a *colleague* - it still assumes you know what a STEMI or an insulin infusion is. The next task keeps the very same note but changes who it is for.

## 4. Task 2 - Explain it for the patient

Same note, same model - **only the system prompt changes**, and the output changes completely. Task 1 wrote for a clinician. This time the reader is the patient: no jargon, no abbreviations, a calm and clear tone, and a plain explanation of what was found and what was done.

The input is *identical* to Task 1. Everything that makes the two outputs so different lives in the system prompt - that is how much control it gives you over who an answer is for.

In [27]:
PATIENT_SYSTEM = (
    "You are helping a patient understand their own medical note. "
    "Rewrite it in warm, plain language that someone with no medical training can follow. "
    "Explain what was found and what was done, and put any medical term into everyday words. "
    "Use 2 to 3 short sentences. Do not add advice, opinions, or any fact that is not in the "
    "note, and do not include names or identifiers."
)

def explain_for_patient(note_text):
    return call_claude(PATIENT_SYSTEM, note_text, max_tokens=160).strip()

notes["patient_explanation"] = notes["text"].apply(explain_for_patient)
notes[["note_id", "llm_summary", "patient_explanation"]]

,note_id,llm_summary,patient_explanation
0,n01,"Acute inferior STEMI treated with aspirin, heparin, and urgent cardiac catheterization.","You came to the hospital with severe chest pain that spread to your left arm, along with sweating and feeling sick. Tests showed signs of a heart attack, specifically in the lower part of your hea..."
1,n02,"COPD exacerbation with hypoxemia treated with bronchodilators, steroids, and supplemental oxygen.","You came in having trouble breathing for the past three days, with wheezing and coughing up greenish mucus. Your oxygen level was low at 86%, and your body was working hard to breathe. The doctors..."
2,n03,"Bleeding gastric ulcer with melena and anemia, treated with PPI infusion and endoscopic clipping.","# What Happened\n\nYou've had a sore, burning spot in your stomach (called an ulcer) that's been bleeding, which is why your stools turned black and tarry. Your blood count was low because of this..."
3,n04,"Acute ischemic stroke with facial droop, arm weakness, and dysarthria; thrombolysis candidate admitted to stroke unit.","You came to the hospital because you suddenly developed drooping on the right side of your face, weakness in your arm, and difficulty speaking about an hour and a half ago—these are signs of a str..."
4,n05,Type 1 diabetic with diabetic ketoacidosis treated with IV fluids and insulin infusion.,"You came in feeling very sick with vomiting and unusual breathing, and you were quite drowsy. Your blood tests showed that your diabetes wasn't controlled—your blood sugar was very high and your b..."
5,n06,"Acute left ureteral calculus with hematuria treated with analgesia, hydration, and urology referral.","You have a small stone (about the size of a grain of rice) stuck in the tube that carries urine from your kidney to your bladder on the left side, which is causing your severe pain. You were given..."
6,n07,Acute appendicitis with peritoneal signs; elevated WBC; surgical consultation obtained.,"You came in with pain that started around your belly button and then moved to the lower right side of your abdomen, along with a fever, feeling sick to your stomach, and tenderness when we pressed..."
7,n08,"68-year-old with community-acquired pneumonia presenting with fever, productive cough, dyspnea, and right lower lobe consolidation.","You came in with a fever, chills, and a cough that's been going on for four days, and you're having trouble catching your breath. The doctor found crackling sounds in your right lung and saw on a ..."


### Same facts, two registers

Put one note's two versions next to each other to feel the gap - identical clinical content, completely different language.

In [28]:
row = notes.loc[0]
print("THE NOTE\n", row["text"], "\n")
print("FOR A CLINICIAN (Task 1)\n", row["llm_summary"], "\n")
print("FOR THE PATIENT (Task 2)\n", row["patient_explanation"])

THE NOTE
 64-year-old man with 2 hours of crushing substernal chest pain radiating to the left arm, with sweating and nausea. ECG shows ST elevation in the inferior leads and troponin is markedly raised. Given aspirin and heparin and taken for urgent cardiac catheterization. 

FOR A CLINICIAN (Task 1)
 Acute inferior STEMI treated with aspirin, heparin, and urgent cardiac catheterization. 

FOR THE PATIENT (Task 2)
 You came to the hospital with severe chest pain that spread to your left arm, along with sweating and feeling sick. Tests showed signs of a heart attack, specifically in the lower part of your heart muscle. You were given blood-thinning medications and taken for an emergency procedure where doctors look at the blood vessels in your heart to find and fix the blockage.


> **This one is a safety matter, not just a style choice.** Patient-facing text is higher-stakes than an internal summary: a model can soften a serious finding, drop a caveat, or sound more reassuring than the facts warrant. Plain-language output meant for patients should be **reviewed by a clinician before the patient sees it**. Easy to understand and safe to send unread are not the same thing.

Both summaries above are still free text. The next two tasks turn notes into values you can actually **compute** on - a category and a flag.

---

## 5. Task 3 - Classify into a department

Now we constrain the output. Instead of letting the model say anything, we give it a **fixed list** and insist it pick exactly one. This is how you turn text into a categorical column with a controlled vocabulary - the kind of column the encoding notebook then one-hot encodes.

In [29]:
ALLOWED_DEPARTMENTS = [
    "Cardiology", "Respiratory", "Gastroenterology",
    "Neurology", "Endocrinology", "Urology",
]

CLASSIFY_SYSTEM = (
    "You are a triage assistant. Read the clinical note and choose the single most "
    "appropriate department from this exact list:\n"
    + ", ".join(ALLOWED_DEPARTMENTS) + ".\n"
    "Reply with ONLY the department name, exactly as written above, and nothing else."
)

def classify(note_text):
    raw = call_claude(CLASSIFY_SYSTEM, note_text, max_tokens=15)
    # models sometimes add punctuation or a trailing line - take the first clean line
    return raw.strip().strip(".").splitlines()[0].strip()

notes["llm_department"] = notes["text"].apply(classify)
notes[["note_id", "department_reference", "llm_department"]]

,note_id,department_reference,llm_department
0,n01,Cardiology,Cardiology
1,n02,Respiratory,Respiratory
2,n03,Gastroenterology,Gastroenterology
3,n04,Neurology,Neurology
4,n05,Endocrinology,Endocrinology
5,n06,Urology,Urology
6,n07,Gastroenterology,Gastroenterology
7,n08,Respiratory,Respiratory
